# Favorita 01 — Data Inventory and Quality

## tl;dr

This diagnostic notebook verifies the local eight-file Favorita boundary and performs memory-safe quality checks. It deliberately separates facts already verified in governance from findings recomputed here. Run the notebook before Notebook 02.

**Safety boundary:** raw files are read-only, raw records are not copied, `train.csv` is never loaded into one DataFrame, and only small previews plus aggregate-only JSON evidence are emitted. This notebook does not approve cleaning rules or research readiness.


## Context & Methods

The intended reader is the MSc researcher and reviewers assessing whether the source is suitable for a later forecasting experiment. The notebook validates inventory, schemas, missingness, uniqueness risks, target edge cases, and master-data coverage.

### Key assumptions

- CSV headers and the raw path are governed by the merged Favorita verification record.
- `id` is assessed as the supplied row identifier; `(date, store_nbr, item_nbr)` is assessed as the candidate observation grain.
- Missing promotion/oil values and negative sales are observations requiring policy decisions, not silently corrected defects.
- Full-file checks are chunked. Retained samples are fixed-seed, bounded, and unsuitable for population estimates unless explicitly labelled.


### 1. Configure bounded execution


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import time

import numpy as np
import pandas as pd

EDA_MODE = "bounded"
TRAIN_CHUNKSIZE = 1_000_000
MAX_SAMPLE_ROWS = 250_000
RANDOM_SEED = 42

if EDA_MODE != "bounded":
    raise ValueError("This research batch permits only EDA_MODE='bounded'.")

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the EDIP repository.")

REPO_ROOT = find_repo_root(Path.cwd())
RAW_DIR = REPO_ROOT / "data" / "raw" / "favorita-grocery-sales-forecasting"
EVIDENCE_DIR = REPO_ROOT / "artifacts" / "reports" / "favorita_eda"
PLOT_DIR = EVIDENCE_DIR / "plots"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = (
    "train.csv",
    "test.csv",
    "stores.csv",
    "items.csv",
    "transactions.csv",
    "holidays_events.csv",
    "oil.csv",
    "sample_submission.csv",
)

SOURCE_URL = "https://www.kaggle.com/competitions/favorita-grocery-sales-forecasting"
ANALYSIS_STARTED_AT = datetime.now(timezone.utc).isoformat()

def show_frame(frame: pd.DataFrame, rows: int = 10) -> None:
    """Print a bounded table that also works outside an interactive kernel."""
    print(frame.head(rows).to_string(index=False))

print({
    "eda_mode": EDA_MODE,
    "train_chunksize": TRAIN_CHUNKSIZE,
    "max_sample_rows": MAX_SAMPLE_ROWS,
    "random_seed": RANDOM_SEED,
    "raw_dir": RAW_DIR.as_posix(),
    "analysis_started_at_utc": ANALYSIS_STARTED_AT,
})


{'eda_mode': 'bounded', 'train_chunksize': 1000000, 'max_sample_rows': 250000, 'random_seed': 42, 'raw_dir': '/mnt/d/my_AI_projects/enterprise_decision_intelligence_platform_EDIP/data/raw/favorita-grocery-sales-forecasting', 'analysis_started_at_utc': '2026-08-03T11:14:34.633191+00:00'}


## Data

### 2. Validate the immutable raw boundary

File sizes and schemas are metadata-only evidence. The notebook does not calculate checksums again because the governance record already contains verified SHA-256 values and a second 5 GB byte pass would add cost without answering this batch's research questions.


In [2]:
missing_files = [name for name in EXPECTED_FILES if not (RAW_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing expected raw files: {missing_files}")

nested_csv_directories = [
    path.relative_to(RAW_DIR).as_posix()
    for path in RAW_DIR.rglob("*.csv")
    if path.is_dir()
]
if nested_csv_directories:
    raise ValueError(f"Unexpected nested .csv directories: {nested_csv_directories}")

inventory_rows = []
for name in EXPECTED_FILES:
    path = RAW_DIR / name
    inventory_rows.append({
        "filename": name,
        "relative_path": path.relative_to(REPO_ROOT).as_posix(),
        "bytes": path.stat().st_size,
        "columns": pd.read_csv(path, nrows=0).columns.tolist(),
    })
inventory = pd.DataFrame(inventory_rows)
show_frame(inventory.assign(columns=inventory["columns"].map(", ".join)), rows=8)


             filename                                                     relative_path      bytes                                                   columns
            train.csv             data/raw/favorita-grocery-sales-forecasting/train.csv 4997452288    id, date, store_nbr, item_nbr, unit_sales, onpromotion
             test.csv              data/raw/favorita-grocery-sales-forecasting/test.csv  126163026                id, date, store_nbr, item_nbr, onpromotion
           stores.csv            data/raw/favorita-grocery-sales-forecasting/stores.csv       1387                     store_nbr, city, state, type, cluster
            items.csv             data/raw/favorita-grocery-sales-forecasting/items.csv     101841                       item_nbr, family, class, perishable
     transactions.csv      data/raw/favorita-grocery-sales-forecasting/transactions.csv    1552637                             date, store_nbr, transactions
  holidays_events.csv   data/raw/favorita-grocery-sales-fo

## 3. Preview the first five rows of each raw CSV


In [3]:
from IPython.display import Markdown, display

PREVIEW_ROWS = 5

for filename in EXPECTED_FILES:
    preview_path = RAW_DIR / filename
    preview_df = pd.read_csv(preview_path, nrows=PREVIEW_ROWS)

    display(Markdown(f"### `{filename}` — first {PREVIEW_ROWS} rows"))
    display(preview_df)

    del preview_df


### `train.csv` — first 5 rows

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


### `test.csv` — first 5 rows

,id,date,store_nbr,item_nbr,onpromotion
0,125497040,2017-08-16,1,96995,False
1,125497041,2017-08-16,1,99197,False
2,125497042,2017-08-16,1,103501,False
3,125497043,2017-08-16,1,103520,False
4,125497044,2017-08-16,1,103665,False


### `stores.csv` — first 5 rows

...

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


### `oil.csv` — first 5 rows

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


### `sample_submission.csv` — first 5 rows

,id,unit_sales
0,125497040,0
1,125497041,0
2,125497042,0
3,125497043,0
4,125497044,0


### 4. Declare selected dtypes and bounded previews


In [3]:
TRAIN_DTYPES = {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float64",
    "onpromotion": "boolean",
}
TEST_DTYPES = {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "onpromotion": "boolean",
}
SMALL_DTYPES = {
    "stores.csv": {"store_nbr": "int16", "city": "string", "state": "string", "type": "category", "cluster": "int16"},
    "items.csv": {"item_nbr": "int32", "family": "category", "class": "int16", "perishable": "int8"},
    "transactions.csv": {"store_nbr": "int16", "transactions": "int32"},
    "holidays_events.csv": {"type": "category", "locale": "category", "locale_name": "string", "description": "string", "transferred": "boolean"},
    "oil.csv": {"dcoilwtico": "float64"},
    "sample_submission.csv": {"id": "int64", "unit_sales": "float64"},
}

dtype_record = {
    "train.csv": TRAIN_DTYPES,
    "test.csv": TEST_DTYPES,
    **SMALL_DTYPES,
}
print(json.dumps(dtype_record, indent=2, default=str))

for name in EXPECTED_FILES:
    preview = pd.read_csv(RAW_DIR / name, nrows=3)
    print(f"\nBounded preview: {name} ({len(preview)} rows)")
    show_frame(preview, rows=3)
    del preview


{
  "train.csv": {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float64",
    "onpromotion": "boolean"
  },
  "test.csv": {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "onpromotion": "boolean"
  },
  "stores.csv": {
    "store_nbr": "int16",
    "city": "string",
    "state": "string",
    "type": "category",
    "cluster": "int16"
  },
  "items.csv": {
    "item_nbr": "int32",
    "family": "category",
    "class": "int16",
    "perishable": "int8"
  },
  "transactions.csv": {
    "store_nbr": "int16",
    "transactions": "int32"
  },
  "holidays_events.csv": {
    "type": "category",
    "locale": "category",
    "locale_name": "string",
    "description": "string",
    "transferred": "boolean"
  },
  "oil.csv": {
    "dcoilwtico": "float64"
  },
  "sample_submission.csv": {
    "id": "int64",
    "unit_sales": "float64"
  }
}

Bounded preview: train.csv (3 rows)
 id       date  store_nbr  item_nbr  unit_sales 

## Results

### 5. Profile small reference files

These files fit comfortably in memory. Duplicate checks distinguish exact rows from candidate business keys. Holiday dates are not assumed unique because multiple locale/event records may legitimately share a date.


In [ ]:
stores = pd.read_csv(RAW_DIR / "stores.csv", dtype=SMALL_DTYPES["stores.csv"])
items = pd.read_csv(RAW_DIR / "items.csv", dtype=SMALL_DTYPES["items.csv"])
transactions = pd.read_csv(RAW_DIR / "transactions.csv", dtype=SMALL_DTYPES["transactions.csv"], parse_dates=["date"])
holidays = pd.read_csv(RAW_DIR / "holidays_events.csv", dtype=SMALL_DTYPES["holidays_events.csv"], parse_dates=["date"])
oil = pd.read_csv(RAW_DIR / "oil.csv", dtype=SMALL_DTYPES["oil.csv"], parse_dates=["date"])

small_quality = pd.DataFrame([
    {"file": "stores.csv", "rows": len(stores), "exact_duplicate_rows": int(stores.duplicated().sum()), "duplicate_candidate_keys": int(stores.duplicated("store_nbr").sum()), "null_cells": int(stores.isna().sum().sum())},
    {"file": "items.csv", "rows": len(items), "exact_duplicate_rows": int(items.duplicated().sum()), "duplicate_candidate_keys": int(items.duplicated("item_nbr").sum()), "null_cells": int(items.isna().sum().sum())},
    {"file": "transactions.csv", "rows": len(transactions), "exact_duplicate_rows": int(transactions.duplicated().sum()), "duplicate_candidate_keys": int(transactions.duplicated(["date", "store_nbr"]).sum()), "null_cells": int(transactions.isna().sum().sum())},
    {"file": "holidays_events.csv", "rows": len(holidays), "exact_duplicate_rows": int(holidays.duplicated().sum()), "duplicate_candidate_keys": int(holidays.duplicated(["date", "type", "locale", "locale_name", "description"]).sum()), "null_cells": int(holidays.isna().sum().sum())},
    {"file": "oil.csv", "rows": len(oil), "exact_duplicate_rows": int(oil.duplicated().sum()), "duplicate_candidate_keys": int(oil.duplicated("date").sum()), "null_cells": int(oil.isna().sum().sum())},
])
show_frame(small_quality, rows=10)

small_nulls = pd.concat({
    "stores.csv": stores.isna().sum(),
    "items.csv": items.isna().sum(),
    "transactions.csv": transactions.isna().sum(),
    "holidays_events.csv": holidays.isna().sum(),
    "oil.csv": oil.isna().sum(),
}, names=["file", "column"]).rename("null_count").reset_index()
show_frame(small_nulls[small_nulls["null_count"] > 0], rows=20)


               file  rows  exact_duplicate_rows  duplicate_candidate_keys  null_cells
         stores.csv    54                     0                         0           0
          items.csv  4100                     0                         0           0
   transactions.csv 83488                     0                         0           0
holidays_events.csv   350                     0                         0           0
            oil.csv  1218                     0                         0          43
   file     column  null_count
oil.csv dcoilwtico          43


### 6. Stream the complete training file

This is a full-file aggregation with bounded memory. Expected cost is roughly 126 chunks and a sequential read of approximately 5 GB. The retained sample is capped at `MAX_SAMPLE_ROWS` and re-sampled with `RANDOM_SEED`; population totals come from all chunks, never from the sample.


In [5]:
train_path = RAW_DIR / "train.csv"
train_nulls = pd.Series(0, index=["id", "date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"], dtype="int64")
promotion_counts = {"true": 0, "false": 0, "missing": 0}
train_stores, train_items = set(), set()
train_rows = negative_sales = zero_sales = 0
sales_sum = 0.0
sales_min, sales_max = math.inf, -math.inf
train_date_min, train_date_max = None, None
id_order_breaks = id_duplicates = 0
composite_order_breaks = composite_duplicates = 0
previous_id = previous_composite = None
sample_parts = []
expected_chunks = math.ceil(125_497_040 / TRAIN_CHUNKSIZE)
sample_per_chunk = max(1, math.ceil(MAX_SAMPLE_ROWS / expected_chunks))
scan_started = time.perf_counter()

reader = pd.read_csv(
    train_path,
    usecols=["id", "date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"],
    dtype=TRAIN_DTYPES,
    chunksize=TRAIN_CHUNKSIZE,
)
for chunk_index, chunk in enumerate(reader):
    train_rows += len(chunk)
    train_nulls = train_nulls.add(chunk.isna().sum().astype("int64"), fill_value=0).astype("int64")
    train_stores.update(int(value) for value in chunk["store_nbr"].unique())
    train_items.update(int(value) for value in chunk["item_nbr"].unique())

    sales = chunk["unit_sales"]
    negative_sales += int((sales < 0).sum())
    zero_sales += int((sales == 0).sum())
    sales_sum += float(sales.sum())
    sales_min = min(sales_min, float(sales.min()))
    sales_max = max(sales_max, float(sales.max()))

    promotion_counts["missing"] += int(chunk["onpromotion"].isna().sum())
    promotion_counts["true"] += int((chunk["onpromotion"] == True).sum())
    promotion_counts["false"] += int((chunk["onpromotion"] == False).sum())

    chunk_date_min, chunk_date_max = chunk["date"].min(), chunk["date"].max()
    train_date_min = chunk_date_min if train_date_min is None else min(train_date_min, chunk_date_min)
    train_date_max = chunk_date_max if train_date_max is None else max(train_date_max, chunk_date_max)

    ids = chunk["id"]
    id_duplicates += int(ids.duplicated().sum())
    id_order_breaks += int((ids.diff().dropna() <= 0).sum())
    if previous_id is not None:
        id_duplicates += int(ids.iloc[0] == previous_id)
        id_order_breaks += int(ids.iloc[0] <= previous_id)
    previous_id = int(ids.iloc[-1])

    composite_index = pd.MultiIndex.from_frame(chunk[["date", "store_nbr", "item_nbr"]])
    composite_duplicates += int(composite_index.duplicated().sum())
    if not composite_index.is_monotonic_increasing:
        composite_order_breaks += 1
    first_composite = tuple(composite_index[0])
    if previous_composite is not None:
        composite_duplicates += int(first_composite == previous_composite)
        composite_order_breaks += int(first_composite < previous_composite)
    previous_composite = tuple(composite_index[-1])

    take = min(sample_per_chunk, len(chunk))
    sample_parts.append(chunk.sample(n=take, random_state=RANDOM_SEED + chunk_index))

    if (chunk_index + 1) % 25 == 0:
        print(f"processed_chunks={chunk_index + 1}, processed_rows={train_rows:,}")

train_sample = pd.concat(sample_parts, ignore_index=True)
if len(train_sample) > MAX_SAMPLE_ROWS:
    train_sample = train_sample.sample(n=MAX_SAMPLE_ROWS, random_state=RANDOM_SEED).sort_values("id")
train_scan_seconds = time.perf_counter() - scan_started

train_summary = {
    "rows": train_rows,
    "date_min": train_date_min,
    "date_max": train_date_max,
    "unique_stores": len(train_stores),
    "unique_items": len(train_items),
    "null_counts": {key: int(value) for key, value in train_nulls.items()},
    "promotion_counts": promotion_counts,
    "negative_unit_sales_rows": negative_sales,
    "zero_unit_sales_rows": zero_sales,
    "positive_unit_sales_rows": train_rows - negative_sales - zero_sales,
    "unit_sales_min": sales_min,
    "unit_sales_max": sales_max,
    "unit_sales_mean": sales_sum / train_rows,
    "id_duplicate_count": id_duplicates,
    "id_order_breaks": id_order_breaks,
    "candidate_grain_duplicate_count": composite_duplicates if composite_order_breaks == 0 else None,
    "candidate_grain_order_break_chunks": composite_order_breaks,
    "bounded_sample_rows": len(train_sample),
    "scan_seconds": round(train_scan_seconds, 3),
}
print(json.dumps(train_summary, indent=2))
print("\nFixed-seed bounded training sample (5 rows; not population evidence):")
show_frame(train_sample, rows=5)
del sample_parts


processed_chunks=25, processed_rows=25,000,000
processed_chunks=50, processed_rows=50,000,000
processed_chunks=75, processed_rows=75,000,000
processed_chunks=100, processed_rows=100,000,000
processed_chunks=125, processed_rows=125,000,000
{
  "rows": 125497040,
  "date_min": "2013-01-01",
  "date_max": "2017-08-15",
  "unique_stores": 54,
  "unique_items": 4036,
  "null_counts": {
    "id": 0,
    "date": 0,
    "store_nbr": 0,
    "item_nbr": 0,
    "unit_sales": 0,
    "onpromotion": 21657651
  },
  "promotion_counts": {
    "true": 7810622,
    "false": 96028767,
    "missing": 21657651
  },
  "negative_unit_sales_rows": 7795,
  "zero_unit_sales_rows": 0,
  "positive_unit_sales_rows": 125489245,
  "unit_sales_min": -15372.0,
  "unit_sales_max": 89440.0,
  "unit_sales_mean": 8.554865268438203,
  "id_duplicate_count": 0,
  "id_order_breaks": 0,
  "candidate_grain_duplicate_count": 0,
  "candidate_grain_order_break_chunks": 0,
  "bounded_sample_rows": 250000,
  "scan_seconds": 70.918
}

### 7. Stream test identifiers and verify master-data coverage

The Kaggle test file is input-only: it has no local ground-truth target. Its 16-day period can inform a candidate horizon but cannot serve as an internal evaluation result.


In [6]:
test_rows = 0
test_nulls = pd.Series(0, index=["id", "date", "store_nbr", "item_nbr", "onpromotion"], dtype="int64")
test_stores, test_items = set(), set()
test_date_min, test_date_max = None, None
test_id_duplicates = test_id_order_breaks = 0
previous_test_id = None

for chunk in pd.read_csv(
    RAW_DIR / "test.csv",
    usecols=["id", "date", "store_nbr", "item_nbr", "onpromotion"],
    dtype=TEST_DTYPES,
    chunksize=TRAIN_CHUNKSIZE,
):
    test_rows += len(chunk)
    test_nulls = test_nulls.add(chunk.isna().sum().astype("int64"), fill_value=0).astype("int64")
    test_stores.update(int(value) for value in chunk["store_nbr"].unique())
    test_items.update(int(value) for value in chunk["item_nbr"].unique())
    test_date_min = chunk["date"].min() if test_date_min is None else min(test_date_min, chunk["date"].min())
    test_date_max = chunk["date"].max() if test_date_max is None else max(test_date_max, chunk["date"].max())
    ids = chunk["id"]
    test_id_duplicates += int(ids.duplicated().sum())
    test_id_order_breaks += int((ids.diff().dropna() <= 0).sum())
    if previous_test_id is not None:
        test_id_duplicates += int(ids.iloc[0] == previous_test_id)
        test_id_order_breaks += int(ids.iloc[0] <= previous_test_id)
    previous_test_id = int(ids.iloc[-1])

master_stores = set(int(value) for value in stores["store_nbr"])
master_items = set(int(value) for value in items["item_nbr"])
transaction_stores = set(int(value) for value in transactions["store_nbr"].unique())
relationship_findings = {
    "train_store_orphans": sorted(train_stores - master_stores),
    "test_store_orphans": sorted(test_stores - master_stores),
    "transaction_store_orphans": sorted(transaction_stores - master_stores),
    "train_item_orphan_count": len(train_items - master_items),
    "test_item_orphan_count": len(test_items - master_items),
    "master_items_not_observed_in_train": len(master_items - train_items),
    "master_items_not_observed_in_test": len(master_items - test_items),
}
test_summary = {
    "rows": test_rows,
    "date_min": test_date_min,
    "date_max": test_date_max,
    "inclusive_days": (pd.Timestamp(test_date_max) - pd.Timestamp(test_date_min)).days + 1,
    "unique_stores": len(test_stores),
    "unique_items": len(test_items),
    "null_counts": {key: int(value) for key, value in test_nulls.items()},
    "id_duplicate_count": test_id_duplicates,
    "id_order_breaks": test_id_order_breaks,
}
print(json.dumps({"test": test_summary, "relationships": relationship_findings}, indent=2))


{
  "test": {
    "rows": 3370464,
    "date_min": "2017-08-16",
    "date_max": "2017-08-31",
    "inclusive_days": 16,
    "unique_stores": 54,
    "unique_items": 3901,
    "null_counts": {
      "id": 0,
      "date": 0,
      "store_nbr": 0,
      "item_nbr": 0,
      "onpromotion": 0
    },
    "id_duplicate_count": 0,
    "id_order_breaks": 0
  },
  "relationships": {
    "train_store_orphans": [],
    "test_store_orphans": [],
    "transaction_store_orphans": [],
    "train_item_orphan_count": 0,
    "test_item_orphan_count": 0,
    "master_items_not_observed_in_train": 64,
    "master_items_not_observed_in_test": 199
  }
}


### 8. Persist small aggregate-only evidence


In [7]:
quality_evidence = {
    "provenance": {
        "notebook": "notebooks/favorita/01_data_inventory_and_quality.ipynb",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_url": SOURCE_URL,
        "source_files": list(EXPECTED_FILES),
        "raw_path": RAW_DIR.relative_to(REPO_ROOT).as_posix(),
        "eda_mode": EDA_MODE,
        "train_chunksize": TRAIN_CHUNKSIZE,
        "max_sample_rows": MAX_SAMPLE_ROWS,
        "random_seed": RANDOM_SEED,
        "contains_raw_records": False,
    },
    "inventory": inventory_rows,
    "train": train_summary,
    "test": test_summary,
    "relationships": relationship_findings,
    "small_file_quality": small_quality.to_dict(orient="records"),
    "oil_missing_price_count": int(oil["dcoilwtico"].isna().sum()),
    "holiday_transferred_count": int(holidays["transferred"].fillna(False).sum()),
}
quality_path = EVIDENCE_DIR / "01_data_inventory_and_quality_summary.json"
quality_path.write_text(json.dumps(quality_evidence, indent=2, default=str), encoding="utf-8")
print(f"Wrote aggregate-only ignored evidence: {quality_path.relative_to(REPO_ROOT).as_posix()}")
print(f"Evidence bytes: {quality_path.stat().st_size:,}")


Wrote aggregate-only ignored evidence: artifacts/reports/favorita_eda/01_data_inventory_and_quality_summary.json
Evidence bytes: 5,109


## Takeaways

- **Verified facts:** the governance record already establishes eight file identities, checksums, 125,497,040 training rows, the 2013-01-01 through 2017-08-15 training period, 54 stores, 4,100 master items, 21,657,651 missing promotion values, 7,795 negative sales rows, and 43 missing oil prices.
- **Notebook-derived checks:** the executed outputs above recompute the required inventory, key ordering, candidate-grain duplication risk, nulls, target edge cases, and child-to-master coverage using the current local files.
- **Analytical risks:** negative sales semantics, missing promotion/oil policy, holiday multiplicity, and absent zero-demand rows remain unresolved. A missing row must not be silently equated with zero demand.
- **Gate:** proceed to descriptive temporal aggregation only. Do not preprocess, engineer model features, or train until the research-scope decisions are approved.
